# Building a Model Comparison Pipeline with Multi-Agent Collaboration

_Authored by: [Faridun Mirzoev](https://huggingface.co/faridunm)_

Choosing the right model from the Hugging Face Hub can be overwhelming — there are thousands of options for any given task. In this notebook, we build an automated model comparison pipeline where multiple AI agents collaborate to search, analyze, and compare models for your specific use case.

We'll use [AG2](https://ag2.ai) to orchestrate three specialized agents:
- A **Scout** that searches the Hub and finds candidate models
- An **Analyst** that extracts technical specs from model cards
- An **Advisor** that compares the candidates and gives a recommendation

Each agent has access to real Hugging Face Hub tools and works with live data — this isn't a simulation.

## Setup

In [ ]:
!pip install "ag2[openai]>=0.11.4,<1.0" huggingface_hub -q

## Connecting to a Hugging Face Model

We'll power our agents with an open-source model hosted on the Hugging Face Inference API. The API provides an OpenAI-compatible endpoint, so we just need to set the `base_url` and use a [HF token](https://huggingface.co/settings/tokens) as the API key.

In [ ]:
import os
from huggingface_hub import get_token
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager, LLMConfig

hf_token = get_token()

llm_config = LLMConfig({
    "model": "Qwen/Qwen2.5-Coder-32B-Instruct",
    "api_key": hf_token,
    "api_type": "openai",
    "base_url": "https://router.huggingface.co/v1",
})

## Building the Hub Tools

Before creating agents, we need to give them the ability to interact with the Hugging Face Hub. We'll create three tools that wrap the `huggingface_hub` API:

1. **search_models** — find models for a given task and query
2. **get_model_details** — extract key specs from a model card (size, license, languages, downloads)
3. **compare_model_stats** — fetch download/like counts for side-by-side comparison

In [ ]:
import json
from typing import Annotated
from huggingface_hub import HfApi, ModelCard

api = HfApi()


def search_models(
    task: Annotated[str, "The task type, e.g. 'text-generation', 'text-classification', 'image-classification'"],
    query: Annotated[str, "Search query to filter models"] = "",
    limit: Annotated[int, "Max number of results"] = 5,
) -> str:
    """Search the Hugging Face Hub for models matching a task and query."""
    models = list(api.list_models(
        pipeline_tag=task,
        search=query,
        sort="downloads",
        limit=limit,
    ))
    if not models:
        return "No models found for this task and query."

    results = []
    for m in models:
        results.append({
            "model_id": m.id,
            "downloads": m.downloads,
            "likes": m.likes,
            "pipeline_tag": m.pipeline_tag,
        })
    return json.dumps(results, indent=2)


def get_model_details(
    model_id: Annotated[str, "The model ID, e.g. 'meta-llama/Llama-3.1-8B-Instruct'"],
) -> str:
    """Get detailed information about a model: size, license, description, tags."""
    try:
        info = api.model_info(model_id)
        try:
            card = ModelCard.load(model_id)
            card_text = card.text[:500] if card.text else "No model card text available."
        except Exception:
            card_text = "Could not load model card."

        details = {
            "model_id": info.id,
            "pipeline_tag": info.pipeline_tag,
            "downloads_last_month": info.downloads,
            "likes": info.likes,
            "license": info.card_data.license if info.card_data and info.card_data.license else "not specified",
            "tags": info.tags[:15],
            "card_summary": card_text,
        }
        return json.dumps(details, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)})


def compare_model_stats(
    model_ids: Annotated[list[str], "List of model IDs to compare"],
) -> str:
    """Compare download and like counts for multiple models side by side."""
    comparison = []
    for model_id in model_ids:
        try:
            info = api.model_info(model_id)
            comparison.append({
                "model_id": info.id,
                "downloads": info.downloads,
                "likes": info.likes,
                "license": info.card_data.license if info.card_data and info.card_data.license else "unknown",
                "pipeline_tag": info.pipeline_tag,
            })
        except Exception as e:
            comparison.append({"model_id": model_id, "error": str(e)})
    return json.dumps(comparison, indent=2)

## Creating the Agent Team

Each agent has a focused role and access to specific tools. The **Scout** finds candidates, the **Analyst** digs into technical details, and the **Advisor** synthesizes everything into a recommendation.

We use AG2's decorator pattern to register which agent can call which tool, and which agent executes the result.

In [ ]:
scout = AssistantAgent(
    name="Scout",
    system_message=(
        "You are a model scout. Your job is to search the Hugging Face Hub "
        "to find candidate models for the user's task. Use the search_models tool "
        "to find the top candidates. Present results clearly with model IDs and "
        "download counts. Only use the tools provided — do not make up model names."
    ),
    llm_config=llm_config,
)

analyst = AssistantAgent(
    name="Analyst",
    system_message=(
        "You are a model analyst. When the Scout has found candidates, use "
        "compare_model_stats to get a side-by-side overview of all candidates at once. "
        "Then use get_model_details on the top 2-3 most promising ones only. "
        "Summarize your findings in a structured comparison table. "
        "Only use the tools provided — do not invent specifications."
    ),
    llm_config=llm_config,
)

advisor = AssistantAgent(
    name="Advisor",
    system_message=(
        "You are a model advisor. Based on the Scout's search results and the "
        "Analyst's detailed comparison, provide a clear recommendation. Consider: "
        "download popularity (community trust), license compatibility, model size, "
        "and suitability for the user's stated use case. "
        "Give a top pick with reasoning, plus an alternative. "
        "End your recommendation with TERMINATE."
    ),
    llm_config=llm_config,
)

executor = UserProxyAgent(
    name="Executor",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: (x.get("content") or "").rstrip().endswith("TERMINATE"),
    code_execution_config=False,
)

# Register tools — Scout and Analyst can call them, Executor runs them
for tool_fn, description in [
    (search_models, "Search the Hugging Face Hub for models matching a task and query"),
    (get_model_details, "Get detailed information about a specific model"),
    (compare_model_stats, "Compare download and like counts for multiple models side by side"),
]:
    executor.register_for_execution()(tool_fn)
    scout.register_for_llm(description=description)(tool_fn)
    analyst.register_for_llm(description=description)(tool_fn)

## Running the Pipeline

Let's ask our agent team to help us choose a text-classification model. The GroupChat manager will coordinate which agent speaks when — the Scout searches first, the Analyst digs deeper, and the Advisor wraps up with a recommendation.

In [ ]:
group_chat = GroupChat(
    agents=[executor, scout, analyst, advisor],
    messages=[],
    max_round=20,
    speaker_selection_method="auto",
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

executor.run(
    manager,
    message=(
        "I need a text-classification model for sentiment analysis on product reviews. "
        "It should be open-source with a permissive license (Apache 2.0 or MIT), "
        "well-maintained (high downloads), and not too large (suitable for fine-tuning "
        "on a single GPU). Find the best options and recommend one."
    ),
).process()

## Reviewing the Conversation

Let's trace how the agents collaborated:

In [ ]:
for msg in group_chat.messages:
    name = msg.get("name", msg.get("role", "unknown"))
    content = msg.get("content", "")
    if content and content.strip():
        print(f"{'='*60}")
        print(f"  {name}:")
        print(f"{content[:600]}")
        print()

## Try It Yourself

Change the query below to explore different tasks — image classification, text generation, translation, or anything else on the Hub:

In [ ]:
# Change this to your own use case!
your_query = (
    "I need an image-classification model for classifying medical X-ray images. "
    "It should have a permissive license and be suitable for fine-tuning."
)

group_chat_2 = GroupChat(
    agents=[executor, scout, analyst, advisor],
    messages=[],
    max_round=20,
    speaker_selection_method="auto",
)

manager_2 = GroupChatManager(groupchat=group_chat_2, llm_config=llm_config)
executor.run(manager_2, message=your_query).process()

# Print the advisor's recommendation
for msg in reversed(group_chat_2.messages):
    if msg.get("name") == "Advisor" and msg.get("content", "").strip():
        print("Advisor's Recommendation:")
        print(msg["content"].replace("TERMINATE", "").strip())
        break

## What We Built

This notebook demonstrated a practical model comparison pipeline where:

1. **The Scout** searched the Hub using task filters and sorted by downloads
2. **The Analyst** pulled model cards and specs to build a structured comparison
3. **The Advisor** synthesized the data into an actionable recommendation

The key idea is that each agent has a focused role with access to real tools — they're not just chatting, they're calling the `huggingface_hub` API and working with live data.

**Extending this pattern:**
- Add a **Benchmarker** agent that runs test inferences to compare latency
- Include a **Cost Estimator** that checks model size vs. your hardware
- Connect to the [Hugging Face Inference Endpoints API](https://huggingface.co/docs/inference-endpoints) for deployment recommendations
- Swap the backbone model — any model on the Inference API works as a drop-in replacement

For more on multi-agent patterns, see the [AG2 documentation](https://docs.ag2.ai).